# NO2 DATA Processing

In [5]:

import pandas as pd
import numpy as np
import ast
from pathlib import Path


# =========================
# 0. Change date/input here
# =========================

YEAR = 2022

BASE_GC_DIR = Path("/Users/yuanbao/Desktop/gc_no2")
TROPOMI_DIR = Path(f"/Users/yuanbao/Desktop/TROPOMI/Data/no2/{YEAR}")

REF_LAT = 51.49
REF_LON = 0.07

YEAR_DIR = BASE_GC_DIR / str(YEAR)

# merged profile already saved in year file 
PROFILE_INPUT = YEAR_DIR / f"all_profiles_{YEAR}.csv"

PROFILE_DATA_DIR = YEAR_DIR / "profile_data"
SMOOTHED_DATA_DIR = YEAR_DIR / "smoothed_data"

PROFILE_DATA_DIR.mkdir(parents=True, exist_ok=True)
SMOOTHED_DATA_DIR.mkdir(parents=True, exist_ok=True)

P0 = 101325.0


# =========================
# 1. Utilities function
# =========================

def get_month_dir(month):
    """
    Automatically identify month
    eg. 1 & 01 
    """
    month_int_dir = YEAR_DIR / str(month)
    month_02_dir = YEAR_DIR / f"{month:02d}"

    if month_int_dir.exists():
        return month_int_dir

    if month_02_dir.exists():
        return month_02_dir

    return month_int_dir


def parse_cloudflag(series):
    """
    Processing cloudflag。
    Accept True, False, true, false, 0, 1。
    """
    if series.dtype == bool:
        return series

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "false": False,
            "true": True,
            "0": False,
            "1": True,
            "nan": np.nan,
            "none": np.nan,
        })
    )


def safe_array_from_string(x):
    """
    Save TROPOMI string list to numpy array。
    """
    if isinstance(x, str):
        return np.array(ast.literal_eval(x), dtype=float)

    if isinstance(x, list):
        return np.array(x, dtype=float)

    return np.array(x, dtype=float)


# =========================
# 2. Generate daily_mean_profile.csv
# =========================

def make_daily_mean_profile():
    if not PROFILE_INPUT.exists():
        raise FileNotFoundError(f"Can't find merged profile file: {PROFILE_INPUT}")

    df_profile = pd.read_csv(PROFILE_INPUT)
    df_profile.columns = df_profile.columns.str.strip()

    if "Date_Time" not in df_profile.columns:
        raise ValueError("Can't fine Date_Time Column in merged profile")

    if "altitude_km" not in df_profile.columns:
        raise ValueError("Can't find altitude_km Column in merged profile")

    df_profile["Date_Time"] = pd.to_datetime(df_profile["Date_Time"], errors="coerce")
    df_profile = df_profile.dropna(subset=["Date_Time"])

    df_profile["date"] = df_profile["Date_Time"].dt.date.astype(str)

    agg_cols = {
        "pressure_hPa": "mean",
        "aer_prf_um2/cm3": "mean",
        "aer_err_um2/cm3": "mean",
        "aer_prf_ext": "mean",
        "aer_err_ext": "mean",
        "no2_prf_molec/cm3": "mean",
        "no2_err_molec/cm3": "mean",
        "no2_prf_partcol": "mean",
        "no2_apriori_partcol": "mean",
        "no2_err_partcol": "mean",
        "no2_prf_ppb": "mean",
        "no2_err_ppb": "mean",
        "no2_apriori_molec/cm3": "mean",
        "apriori_aerarea": "mean",
    }

    existing_agg_cols = {
        col: method
        for col, method in agg_cols.items()
        if col in df_profile.columns
    }

    if len(existing_agg_cols) == 0:
        raise ValueError("Can't find any profile column in daily mean")

    daily_profile = (
        df_profile
        .groupby(["date", "altitude_km"], as_index=False)
        .agg(existing_agg_cols)
        .sort_values(["date", "altitude_km"])
        .reset_index(drop=True)
    )

    if "time_str" in df_profile.columns:
        n_profiles_per_day = (
            df_profile[["date", "time_str"]]
            .drop_duplicates()
            .groupby("date")
            .size()
            .reset_index(name="n_profiles")
        )
    else:
        n_profiles_per_day = (
            df_profile
            .groupby("date")
            .size()
            .reset_index(name="n_profiles")
        )

    daily_profile = daily_profile.merge(n_profiles_per_day, on="date", how="left")

    out_file = PROFILE_DATA_DIR / "daily_mean_profile.csv"
    daily_profile.to_csv(out_file, index=False)

    print(f"Saved: {out_file}")
    return daily_profile


# =========================
# 3. Generate daily_column_from_profile.csv
# =========================

def make_daily_column_from_profile(daily_profile):
    if "no2_prf_partcol" not in daily_profile.columns:
        print("Skip daily_column_from_profile.csv because no2_prf_partcol is missing")
        return None

    daily_one_value = (
        daily_profile
        .groupby("date", as_index=False)["no2_prf_partcol"]
        .sum()
        .rename(columns={"no2_prf_partcol": "daily_column_from_profile"})
    )

    out_file = PROFILE_DATA_DIR / "daily_column_from_profile.csv"
    daily_one_value.to_csv(out_file, index=False)

    print(f"Saved: {out_file}")
    return daily_one_value


# =========================
# 4. Generate maxdoas_profile_for_smoothing.csv
# =========================

def make_profile_for_smoothing(daily_profile):
    keep_cols = [
        "date",
        "altitude_km",
        "pressure_hPa",
        "no2_prf_molec/cm3",
        "no2_apriori_molec/cm3",
    ]

    missing = [c for c in keep_cols if c not in daily_profile.columns]

    if missing:
        raise ValueError(f"Can't find column in daily_mean_profile: {missing}")

    df_out = (
        daily_profile[keep_cols]
        .copy()
        .sort_values(["date", "altitude_km"])
        .reset_index(drop=True)
    )

    out_file = PROFILE_DATA_DIR / "maxdoas_profile_for_smoothing.csv"
    df_out.to_csv(out_file, index=False)

    print(f"Saved: {out_file}")
    return df_out


# =========================
# 5. MAXDOAS hourly to daily summary
# =========================

def process_one_timeseries_file(file):
    df = pd.read_csv(file)
    df.columns = df.columns.str.strip()

    required_cols = [
        "Date_Time",
        "no2_column_molec/cm2",
        "no2_column_err_molec/cm2",
        "no2_surf_ppb",
        "no2_surf_err_ppb",
        "cloudflag",
    ]

    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        print(f"Skip {file.name}, missing columns: {missing}")
        return None

    df["Date_Time"] = pd.to_datetime(df["Date_Time"], errors="coerce")

    df = df[required_cols].copy()
    df["cloudflag"] = parse_cloudflag(df["cloudflag"])

    df = df.dropna(subset=required_cols)

    df = df[df["cloudflag"] == False].copy()

    M = len(df)

    if M == 0:
        return None

    daily_mean_column = df["no2_column_molec/cm2"].mean()
    daily_err_column = np.sqrt(np.sum(df["no2_column_err_molec/cm2"] ** 2)) / M

    daily_mean_surface = df["no2_surf_ppb"].mean()
    daily_err_surface = np.sqrt(np.sum(df["no2_surf_err_ppb"] ** 2)) / M

    result = {
        "date": df["Date_Time"].dt.date.iloc[0].strftime("%Y-%m-%d"),
        "daily_mean_no2_column_molec/cm2": daily_mean_column,
        "daily_err_no2_column_molec/cm2": daily_err_column,
        "daily_mean_no2_surface_ppb": daily_mean_surface,
        "daily_err_no2_surface_ppb": daily_err_surface,
        "n_obs": M,
        "file_name": file.name,
    }

    return result


def make_monthly_daily_summary(month):
    month_dir = get_month_dir(month)

    if not month_dir.exists():
        msg = f"MAXDOAS month folder missing: {month_dir}"
        print(f"Skip month {month:02d}: {msg}")
        return None, msg

    rows = []

    for file in month_dir.rglob("UCL_o4_no2_*_timeseries_results.csv"):
        res = process_one_timeseries_file(file)

        if res is not None:
            rows.append(res)

    if len(rows) == 0:
        msg = "No valid MAXDOAS timeseries files"
        print(f"Skip month {month:02d}: {msg}")
        return None, msg

    daily_summary = (
        pd.DataFrame(rows)
        .sort_values("date")
        .reset_index(drop=True)
    )

    out_file = month_dir / f"NO2_{YEAR}_{month:02d}_daily_summary.csv"
    daily_summary.to_csv(out_file, index=False)

    print(f"Saved: {out_file}")
    return daily_summary, "ok"


# =========================
# 6. Read TROPOMI Monthly File
# =========================

def read_tropomi_month(month):
    tropomi_file = TROPOMI_DIR / f"london_no2_{YEAR}{month:02d}.csv"

    if not tropomi_file.exists():
        return None, f"TROPOMI file missing: {tropomi_file}"

    try:
        tropomi = pd.read_csv(tropomi_file)
    except pd.errors.EmptyDataError:
        return None, f"TROPOMI file is empty: {tropomi_file}"

    if len(tropomi) == 0:
        return None, f"TROPOMI file has zero rows: {tropomi_file}"

    tropomi.columns = tropomi.columns.str.strip()

    required_cols = [
        "time_start",
        "latitude",
        "longitude",
        "no2_trop_column",
        "surface_pressure",
        "averaging_kernel",
        "tm5_constant_a",
        "tm5_constant_b",
    ]

    missing = [c for c in required_cols if c not in tropomi.columns]

    if missing:
        return None, f"TROPOMI missing columns: {missing}"

    tropomi["time_start"] = pd.to_datetime(tropomi["time_start"], errors="coerce")
    tropomi = tropomi.dropna(subset=["time_start"])

    tropomi = tropomi.dropna(subset=[
        "latitude",
        "longitude",
        "no2_trop_column",
        "surface_pressure",
        "averaging_kernel",
        "tm5_constant_a",
        "tm5_constant_b",
    ])

    if len(tropomi) == 0:
        return None, "TROPOMI has no valid rows after dropping NaN"

    return tropomi, "ok"


# =========================
# 7. Calculate daily smoothed column
# =========================

def calc_one_day(day_str, tropomi, maxdoas_profile, summary):
    day = pd.Timestamp(day_str).date()

    g = tropomi[tropomi["time_start"].dt.date == day].copy()

    if len(g) == 0:
        return None

    g["dist2"] = (g["latitude"] - REF_LAT) ** 2 + (g["longitude"] - REF_LON) ** 2
    best = g.sort_values("dist2").iloc[0]

    try:
        tropomi_column = float(best["no2_trop_column"])
        sp = float(best["surface_pressure"])

        ak = safe_array_from_string(best["averaging_kernel"])
        a = safe_array_from_string(best["tm5_constant_a"])
        b = safe_array_from_string(best["tm5_constant_b"])
    except Exception:
        return None

    if len(ak) == 0 or len(a) == 0 or len(b) == 0:
        return None

    try:
        p_edge = a + b * sp
        p_edge_35 = np.r_[p_edge[0], p_edge[1::2]]

        p_mid_34 = 0.5 * (p_edge_35[:-1] + p_edge_35[1:])

        z_mid_km = 44330.0 * (1.0 - (p_mid_34 / P0) ** 0.1903) / 1000.0
        z_edge_km = 44330.0 * (1.0 - (p_edge_35 / P0) ** 0.1903) / 1000.0

        layer_thickness_cm = (z_edge_km[1:] - z_edge_km[:-1]) * 1e5
    except Exception:
        return None

    n_layer = min(len(ak), len(z_mid_km), len(layer_thickness_cm))

    if n_layer == 0:
        return None

    ak = ak[:n_layer]
    z_mid_km = z_mid_km[:n_layer]
    layer_thickness_cm = layer_thickness_cm[:n_layer]

    m = maxdoas_profile[maxdoas_profile["date"] == day_str].copy()

    if len(m) == 0:
        return None

    z_max = m["altitude_km"].to_numpy(dtype=float)
    no2_max = m["no2_prf_molec/cm3"].to_numpy(dtype=float)

    valid = np.isfinite(z_max) & np.isfinite(no2_max)

    z_max = z_max[valid]
    no2_max = no2_max[valid]

    if len(z_max) < 2:
        return None

    idx = np.argsort(z_max)
    z_max = z_max[idx]
    no2_max = no2_max[idx]

    max_height = z_max.max()
    mask_valid = z_mid_km <= max_height

    if mask_valid.sum() == 0:
        return None

    no2_full = np.empty(len(z_mid_km))

    no2_full[mask_valid] = np.interp(
        z_mid_km[mask_valid],
        z_max,
        no2_max
    )

    fill_value = no2_full[mask_valid][-1]
    no2_full[~mask_valid] = fill_value

    smoothed_profile = ak * no2_full
    smoothed_column = np.sum(smoothed_profile * layer_thickness_cm)

    s = summary[summary["date"] == day_str]

    if len(s) == 0:
        maxdoas_daily_mean = np.nan
        maxdoas_daily_err = np.nan
        smoothed_err = np.nan
    else:
        row = s.iloc[0]

        maxdoas_daily_mean = float(row["daily_mean_no2_column_molec/cm2"])
        maxdoas_daily_err = float(row["daily_err_no2_column_molec/cm2"])

        if np.isfinite(maxdoas_daily_mean) and maxdoas_daily_mean != 0:
            rel_err = maxdoas_daily_err / maxdoas_daily_mean
            smoothed_err = smoothed_column * rel_err
        else:
            smoothed_err = np.nan

    return {
        "date": day_str,
        "tropomi_time": best["time_start"],
        "pixel_lat": float(best["latitude"]),
        "pixel_lon": float(best["longitude"]),
        "tropomi_column": tropomi_column,
        "maxdoas_daily_mean": maxdoas_daily_mean,
        "maxdoas_daily_err": maxdoas_daily_err,
        "smoothed_column": smoothed_column,
        "smoothed_err": smoothed_err,
    }


# =========================
# 8. Calculate monthly smoothed data
# =========================

def make_monthly_smoothed_data(month, maxdoas_profile, summary):
    tropomi, msg = read_tropomi_month(month)

    if tropomi is None:
        print(f"Skip smoothed month {month:02d}: {msg}")
        return None, msg

    start = f"{YEAR}-{month:02d}-01"
    end = pd.Timestamp(start) + pd.offsets.MonthEnd(0)

    days = pd.date_range(start, end, freq="D")

    results = []
    skipped_days = []

    for d in days:
        day_str = d.strftime("%Y-%m-%d")

        out = calc_one_day(
            day_str=day_str,
            tropomi=tropomi,
            maxdoas_profile=maxdoas_profile,
            summary=summary,
        )

        if out is not None:
            results.append(out)
        else:
            skipped_days.append(day_str)

    if len(results) == 0:
        msg = f"No valid smoothed result for month {month:02d}"
        print(f"Skip smoothed month {month:02d}: {msg}")
        return None, msg

    result_df = pd.DataFrame(results)

    out_file = SMOOTHED_DATA_DIR / f"maxdoas_smoothed_{YEAR}{month:02d}.csv"
    result_df.to_csv(out_file, index=False)

    print(f"Saved: {out_file}")
    print(f"Month {month:02d}: {len(results)} days saved, {len(skipped_days)} days skipped")

    return result_df, "ok"


# =========================
# 9. Main function
# =========================

def main():
    print(f"Start processing YEAR = {YEAR}")
    print(f"Profile input: {PROFILE_INPUT}")
    print(f"TROPOMI dir: {TROPOMI_DIR}")

    processing_log = []
    all_smoothed = []

    daily_profile = make_daily_mean_profile()

    make_daily_column_from_profile(daily_profile)

    maxdoas_profile = make_profile_for_smoothing(daily_profile)
    maxdoas_profile["date"] = maxdoas_profile["date"].astype(str)

    for month in range(1, 13):
        print("")
        print(f"Processing month {month:02d}")

        summary, summary_msg = make_monthly_daily_summary(month)

        if summary is None:
            processing_log.append({
                "year": YEAR,
                "month": f"{month:02d}",
                "daily_summary_status": "skipped",
                "smoothed_status": "skipped",
                "reason": summary_msg,
            })

            continue

        summary["date"] = summary["date"].astype(str)

        smoothed, smoothed_msg = make_monthly_smoothed_data(
            month=month,
            maxdoas_profile=maxdoas_profile,
            summary=summary,
        )

        if smoothed is None:
            processing_log.append({
                "year": YEAR,
                "month": f"{month:02d}",
                "daily_summary_status": "success",
                "smoothed_status": "skipped",
                "reason": smoothed_msg,
            })

            continue

        all_smoothed.append(smoothed)

        processing_log.append({
            "year": YEAR,
            "month": f"{month:02d}",
            "daily_summary_status": "success",
            "smoothed_status": "success",
            "reason": "ok",
        })

    if len(all_smoothed) > 0:
        all_year = pd.concat(all_smoothed, ignore_index=True)

        out_file = SMOOTHED_DATA_DIR / f"maxdoas_smoothed_{YEAR}_all_months.csv"
        all_year.to_csv(out_file, index=False)

        print("")
        print(f"Saved yearly combined file: {out_file}")
    else:
        print("")
        print("No smoothed data generated for the whole year")

    log_df = pd.DataFrame(processing_log)

    log_file = SMOOTHED_DATA_DIR / f"processing_log_{YEAR}.csv"
    log_df.to_csv(log_file, index=False)

    print(f"Saved log: {log_file}")
    print("")
    print("Done")


if __name__ == "__main__":
    main()

Start processing YEAR = 2022
Profile input: /Users/yuanbao/Desktop/gc_no2/2022/all_profiles_2022.csv
TROPOMI dir: /Users/yuanbao/Desktop/TROPOMI/Data/no2/2022
Saved: /Users/yuanbao/Desktop/gc_no2/2022/profile_data/daily_mean_profile.csv
Saved: /Users/yuanbao/Desktop/gc_no2/2022/profile_data/daily_column_from_profile.csv
Saved: /Users/yuanbao/Desktop/gc_no2/2022/profile_data/maxdoas_profile_for_smoothing.csv

Processing month 01
Skip month 01: MAXDOAS month folder missing: /Users/yuanbao/Desktop/gc_no2/2022/1

Processing month 02
Skip month 02: MAXDOAS month folder missing: /Users/yuanbao/Desktop/gc_no2/2022/2

Processing month 03
Skip month 03: MAXDOAS month folder missing: /Users/yuanbao/Desktop/gc_no2/2022/3

Processing month 04
Skip month 04: MAXDOAS month folder missing: /Users/yuanbao/Desktop/gc_no2/2022/4

Processing month 05
Skip month 05: MAXDOAS month folder missing: /Users/yuanbao/Desktop/gc_no2/2022/5

Processing month 06
Skip month 06: MAXDOAS month folder missing: /Users/y

# HCHO Data Processing

In [7]:

import pandas as pd
import numpy as np
import ast
from pathlib import Path


# =========================
# 0. Change date/input here
# =========================

YEAR = 2022

BASE_GC_DIR = Path("/Users/yuanbao/Desktop/nogc_hcho")
TROPOMI_DIR = Path(f"/Users/yuanbao/Desktop/TROPOMI/Data/hcho/{YEAR}")

REF_LAT = 51.49
REF_LON = 0.07

YEAR_DIR = BASE_GC_DIR / str(YEAR)

# merged profile already saved in year file 
PROFILE_INPUT = YEAR_DIR / f"all_profiles_{YEAR}.csv"

PROFILE_DATA_DIR = YEAR_DIR / "profile_data"
SMOOTHED_DATA_DIR = YEAR_DIR / "smoothed_data"

PROFILE_DATA_DIR.mkdir(parents=True, exist_ok=True)
SMOOTHED_DATA_DIR.mkdir(parents=True, exist_ok=True)

P0 = 101325.0


# =========================
# 1. Utilities function
# =========================

def get_month_dir(month):
    """
    Automatically identify month
    eg. 1 & 01 
    """
    month_int_dir = YEAR_DIR / str(month)
    month_02_dir = YEAR_DIR / f"{month:02d}"

    if month_int_dir.exists():
        return month_int_dir

    if month_02_dir.exists():
        return month_02_dir

    return month_int_dir


def parse_cloudflag(series):
    """
    Processing cloudflag。
    Accept True, False, true, false, 0, 1。
    """
    if series.dtype == bool:
        return series

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "false": False,
            "true": True,
            "0": False,
            "1": True,
            "nan": np.nan,
            "none": np.nan,
        })
    )


def safe_array_from_string(x):
    """
    Save TROPOMI string list to numpy array。
    """
    if isinstance(x, str):
        return np.array(ast.literal_eval(x), dtype=float)

    if isinstance(x, list):
        return np.array(x, dtype=float)

    return np.array(x, dtype=float)


# =========================
# 2. Generate daily_mean_profile.csv
# =========================

def make_daily_mean_profile():
    if not PROFILE_INPUT.exists():
        raise FileNotFoundError(f"Can't find merged profile file: {PROFILE_INPUT}")

    df_profile = pd.read_csv(PROFILE_INPUT)
    df_profile.columns = df_profile.columns.str.strip()

    if "Date_Time" not in df_profile.columns:
        raise ValueError("Can't fine Date_Time Column in merged profile")

    if "altitude_km" not in df_profile.columns:
        raise ValueError("Can't find altitude_km Column in merged profile")

    df_profile["Date_Time"] = pd.to_datetime(df_profile["Date_Time"], errors="coerce")
    df_profile = df_profile.dropna(subset=["Date_Time"])

    df_profile["date"] = df_profile["Date_Time"].dt.date.astype(str)

    agg_cols = {
        "pressure_hPa": "mean",
        "aer_prf_um2/cm3": "mean",
        "aer_err_um2/cm3": "mean",
        "aer_prf_ext": "mean",
        "aer_err_ext": "mean",
        "hcho_prf_molec/cm3": "mean",
        "hcho_err_molec/cm3": "mean",
        "hcho_prf_partcol": "mean",
        "hcho_apriori_partcol": "mean",
        "hcho_err_partcol": "mean",
        "hcho_prf_ppb": "mean",
        "hcho_err_ppb": "mean",
        "hcho_apriori_molec/cm3": "mean",
        "apriori_aerarea": "mean",
    }

    existing_agg_cols = {
        col: method
        for col, method in agg_cols.items()
        if col in df_profile.columns
    }

    if len(existing_agg_cols) == 0:
        raise ValueError("Can't find any profile column in daily mean")

    daily_profile = (
        df_profile
        .groupby(["date", "altitude_km"], as_index=False)
        .agg(existing_agg_cols)
        .sort_values(["date", "altitude_km"])
        .reset_index(drop=True)
    )

    if "time_str" in df_profile.columns:
        n_profiles_per_day = (
            df_profile[["date", "time_str"]]
            .drop_duplicates()
            .groupby("date")
            .size()
            .reset_index(name="n_profiles")
        )
    else:
        n_profiles_per_day = (
            df_profile
            .groupby("date")
            .size()
            .reset_index(name="n_profiles")
        )

    daily_profile = daily_profile.merge(n_profiles_per_day, on="date", how="left")

    out_file = PROFILE_DATA_DIR / "daily_mean_profile.csv"
    daily_profile.to_csv(out_file, index=False)

    print(f"Saved: {out_file}")
    return daily_profile


# =========================
# 3. Generate daily_column_from_profile.csv
# =========================

def make_daily_column_from_profile(daily_profile):
    if "hcho_prf_partcol" not in daily_profile.columns:
        print("Skip daily_column_from_profile.csv because hcho_prf_partcol is missing")
        return None

    daily_one_value = (
        daily_profile
        .groupby("date", as_index=False)["hcho_prf_partcol"]
        .sum()
        .rename(columns={"hcho_prf_partcol": "daily_column_from_profile"})
    )

    out_file = PROFILE_DATA_DIR / "daily_column_from_profile.csv"
    daily_one_value.to_csv(out_file, index=False)

    print(f"Saved: {out_file}")
    return daily_one_value


# =========================
# 4. Generate maxdoas_profile_for_smoothing.csv
# =========================

def make_profile_for_smoothing(daily_profile):
    keep_cols = [
        "date",
        "altitude_km",
        "pressure_hPa",
        "hcho_prf_molec/cm3",
        "hcho_apriori_molec/cm3",
    ]

    missing = [c for c in keep_cols if c not in daily_profile.columns]

    if missing:
        raise ValueError(f"Can't find column in daily_mean_profile: {missing}")

    df_out = (
        daily_profile[keep_cols]
        .copy()
        .sort_values(["date", "altitude_km"])
        .reset_index(drop=True)
    )

    out_file = PROFILE_DATA_DIR / "maxdoas_profile_for_smoothing.csv"
    df_out.to_csv(out_file, index=False)

    print(f"Saved: {out_file}")
    return df_out


# =========================
# 5. MAXDOAS hourly to daily summary
# =========================

def process_one_timeseries_file(file):
    df = pd.read_csv(file)
    df.columns = df.columns.str.strip()

    required_cols = [
        "Date_Time",
        "hcho_column_molec/cm2",
        "hcho_column_err_molec/cm2",
        "hcho_surf_ppb",
        "hcho_surf_err_ppb",
        "cloudflag",
    ]

    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        print(f"Skip {file.name}, missing columns: {missing}")
        return None

    df["Date_Time"] = pd.to_datetime(df["Date_Time"], errors="coerce")

    df = df[required_cols].copy()
    df["cloudflag"] = parse_cloudflag(df["cloudflag"])

    df = df.dropna(subset=required_cols)

    df = df[df["cloudflag"] == False].copy()

    M = len(df)

    if M == 0:
        return None

    daily_mean_column = df["hcho_column_molec/cm2"].mean()
    daily_err_column = np.sqrt(np.sum(df["hcho_column_err_molec/cm2"] ** 2)) / M

    daily_mean_surface = df["hcho_surf_ppb"].mean()
    daily_err_surface = np.sqrt(np.sum(df["hcho_surf_err_ppb"] ** 2)) / M

    result = {
        "date": df["Date_Time"].dt.date.iloc[0].strftime("%Y-%m-%d"),
        "daily_mean_hcho_column_molec/cm2": daily_mean_column,
        "daily_err_hcho_column_molec/cm2": daily_err_column,
        "daily_mean_hcho_surface_ppb": daily_mean_surface,
        "daily_err_hcho_surface_ppb": daily_err_surface,
        "n_obs": M,
        "file_name": file.name,
    }

    return result


def make_monthly_daily_summary(month):
    month_dir = get_month_dir(month)

    if not month_dir.exists():
        msg = f"MAXDOAS month folder missing: {month_dir}"
        print(f"Skip month {month:02d}: {msg}")
        return None, msg

    rows = []

    for file in month_dir.rglob("UCL_o4_hcho_*_timeseries_results.csv"):
        res = process_one_timeseries_file(file)

        if res is not None:
            rows.append(res)

    if len(rows) == 0:
        msg = "No valid MAXDOAS timeseries files"
        print(f"Skip month {month:02d}: {msg}")
        return None, msg

    daily_summary = (
        pd.DataFrame(rows)
        .sort_values("date")
        .reset_index(drop=True)
    )

    out_file = month_dir / f"HCHO_{YEAR}_{month:02d}_daily_summary.csv"
    daily_summary.to_csv(out_file, index=False)

    print(f"Saved: {out_file}")
    return daily_summary, "ok"


# =========================
# 6. Read TROPOMI Monthly File
# =========================

def read_tropomi_month(month):
    tropomi_file = TROPOMI_DIR / f"london_hcho_{YEAR}{month:02d}.csv"

    if not tropomi_file.exists():
        return None, f"TROPOMI file missing: {tropomi_file}"

    try:
        tropomi = pd.read_csv(tropomi_file)
    except pd.errors.EmptyDataError:
        return None, f"TROPOMI file is empty: {tropomi_file}"

    if len(tropomi) == 0:
        return None, f"TROPOMI file has zero rows: {tropomi_file}"

    tropomi.columns = tropomi.columns.str.strip()

    required_cols = [
        "time_start",
        "latitude",
        "longitude",
        "hcho_trop_column",
        "surface_pressure",
        "averaging_kernel",
        "tm5_constant_a",
        "tm5_constant_b",
    ]

    missing = [c for c in required_cols if c not in tropomi.columns]

    if missing:
        return None, f"TROPOMI missing columns: {missing}"

    tropomi["time_start"] = pd.to_datetime(tropomi["time_start"], errors="coerce")
    tropomi = tropomi.dropna(subset=["time_start"])

    tropomi = tropomi.dropna(subset=[
        "latitude",
        "longitude",
        "hcho_trop_column",
        "surface_pressure",
        "averaging_kernel",
        "tm5_constant_a",
        "tm5_constant_b",
    ])

    if len(tropomi) == 0:
        return None, "TROPOMI has no valid rows after dropping NaN"

    return tropomi, "ok"


# =========================
# 7. Calculate daily smoothed column
# =========================

def calc_one_day(day_str, tropomi, maxdoas_profile, summary):
    day = pd.Timestamp(day_str).date()

    g = tropomi[tropomi["time_start"].dt.date == day].copy()

    if len(g) == 0:
        return None

    g["dist2"] = (g["latitude"] - REF_LAT) ** 2 + (g["longitude"] - REF_LON) ** 2
    best = g.sort_values("dist2").iloc[0]

    try:
        tropomi_column = float(best["hcho_trop_column"])
        sp = float(best["surface_pressure"])

        ak = safe_array_from_string(best["averaging_kernel"])
        a = safe_array_from_string(best["tm5_constant_a"])
        b = safe_array_from_string(best["tm5_constant_b"])
    except Exception:
        return None

    if len(ak) == 0 or len(a) == 0 or len(b) == 0:
        return None

    try:
        p_edge = a + b * sp
        p_edge_35 = np.r_[p_edge[0], p_edge[1::2]]

        p_mid_34 = 0.5 * (p_edge_35[:-1] + p_edge_35[1:])

        z_mid_km = 44330.0 * (1.0 - (p_mid_34 / P0) ** 0.1903) / 1000.0
        z_edge_km = 44330.0 * (1.0 - (p_edge_35 / P0) ** 0.1903) / 1000.0

        layer_thickness_cm = (z_edge_km[1:] - z_edge_km[:-1]) * 1e5
    except Exception:
        return None

    n_layer = min(len(ak), len(z_mid_km), len(layer_thickness_cm))

    if n_layer == 0:
        return None

    ak = ak[:n_layer]
    z_mid_km = z_mid_km[:n_layer]
    layer_thickness_cm = layer_thickness_cm[:n_layer]

    m = maxdoas_profile[maxdoas_profile["date"] == day_str].copy()

    if len(m) == 0:
        return None

    z_max = m["altitude_km"].to_numpy(dtype=float)
    hcho_max = m["hcho_prf_molec/cm3"].to_numpy(dtype=float)

    valid = np.isfinite(z_max) & np.isfinite(hcho_max)

    z_max = z_max[valid]
    hcho_max = hcho_max[valid]

    if len(z_max) < 2:
        return None

    idx = np.argsort(z_max)
    z_max = z_max[idx]
    hcho_max = hcho_max[idx]

    max_height = z_max.max()
    mask_valid = z_mid_km <= max_height

    if mask_valid.sum() == 0:
        return None

    hcho_full = np.empty(len(z_mid_km))

    hcho_full[mask_valid] = np.interp(
        z_mid_km[mask_valid],
        z_max,
        hcho_max
    )

    fill_value = hcho_full[mask_valid][-1]
    hcho_full[~mask_valid] = fill_value

    smoothed_profile = ak * hcho_full
    smoothed_column = np.sum(smoothed_profile * layer_thickness_cm)

    s = summary[summary["date"] == day_str]

    if len(s) == 0:
        maxdoas_daily_mean = np.nan
        maxdoas_daily_err = np.nan
        smoothed_err = np.nan
    else:
        row = s.iloc[0]

        maxdoas_daily_mean = float(row["daily_mean_hcho_column_molec/cm2"])
        maxdoas_daily_err = float(row["daily_err_hcho_column_molec/cm2"])

        if np.isfinite(maxdoas_daily_mean) and maxdoas_daily_mean != 0:
            rel_err = maxdoas_daily_err / maxdoas_daily_mean
            smoothed_err = smoothed_column * rel_err
        else:
            smoothed_err = np.nan

    return {
        "date": day_str,
        "tropomi_time": best["time_start"],
        "pixel_lat": float(best["latitude"]),
        "pixel_lon": float(best["longitude"]),
        "tropomi_column": tropomi_column,
        "maxdoas_daily_mean": maxdoas_daily_mean,
        "maxdoas_daily_err": maxdoas_daily_err,
        "smoothed_column": smoothed_column,
        "smoothed_err": smoothed_err,
    }


# =========================
# 8. Calculate monthly smoothed data
# =========================

def make_monthly_smoothed_data(month, maxdoas_profile, summary):
    tropomi, msg = read_tropomi_month(month)

    if tropomi is None:
        print(f"Skip smoothed month {month:02d}: {msg}")
        return None, msg

    start = f"{YEAR}-{month:02d}-01"
    end = pd.Timestamp(start) + pd.offsets.MonthEnd(0)

    days = pd.date_range(start, end, freq="D")

    results = []
    skipped_days = []

    for d in days:
        day_str = d.strftime("%Y-%m-%d")

        out = calc_one_day(
            day_str=day_str,
            tropomi=tropomi,
            maxdoas_profile=maxdoas_profile,
            summary=summary,
        )

        if out is not None:
            results.append(out)
        else:
            skipped_days.append(day_str)

    if len(results) == 0:
        msg = f"No valid smoothed result for month {month:02d}"
        print(f"Skip smoothed month {month:02d}: {msg}")
        return None, msg

    result_df = pd.DataFrame(results)

    out_file = SMOOTHED_DATA_DIR / f"maxdoas_smoothed_{YEAR}{month:02d}.csv"
    result_df.to_csv(out_file, index=False)

    print(f"Saved: {out_file}")
    print(f"Month {month:02d}: {len(results)} days saved, {len(skipped_days)} days skipped")

    return result_df, "ok"


# =========================
# 9. Main function
# =========================

def main():
    print(f"Start processing YEAR = {YEAR}")
    print(f"Profile input: {PROFILE_INPUT}")
    print(f"TROPOMI dir: {TROPOMI_DIR}")

    processing_log = []
    all_smoothed = []

    daily_profile = make_daily_mean_profile()

    make_daily_column_from_profile(daily_profile)

    maxdoas_profile = make_profile_for_smoothing(daily_profile)
    maxdoas_profile["date"] = maxdoas_profile["date"].astype(str)

    for month in range(1, 13):
        print("")
        print(f"Processing month {month:02d}")

        summary, summary_msg = make_monthly_daily_summary(month)

        if summary is None:
            processing_log.append({
                "year": YEAR,
                "month": f"{month:02d}",
                "daily_summary_status": "skipped",
                "smoothed_status": "skipped",
                "reason": summary_msg,
            })

            continue

        summary["date"] = summary["date"].astype(str)

        smoothed, smoothed_msg = make_monthly_smoothed_data(
            month=month,
            maxdoas_profile=maxdoas_profile,
            summary=summary,
        )

        if smoothed is None:
            processing_log.append({
                "year": YEAR,
                "month": f"{month:02d}",
                "daily_summary_status": "success",
                "smoothed_status": "skipped",
                "reason": smoothed_msg,
            })

            continue

        all_smoothed.append(smoothed)

        processing_log.append({
            "year": YEAR,
            "month": f"{month:02d}",
            "daily_summary_status": "success",
            "smoothed_status": "success",
            "reason": "ok",
        })

    if len(all_smoothed) > 0:
        all_year = pd.concat(all_smoothed, ignore_index=True)

        out_file = SMOOTHED_DATA_DIR / f"maxdoas_smoothed_{YEAR}_all_months.csv"
        all_year.to_csv(out_file, index=False)

        print("")
        print(f"Saved yearly combined file: {out_file}")
    else:
        print("")
        print("No smoothed data generated for the whole year")

    log_df = pd.DataFrame(processing_log)

    log_file = SMOOTHED_DATA_DIR / f"processing_log_{YEAR}.csv"
    log_df.to_csv(log_file, index=False)

    print(f"Saved log: {log_file}")
    print("")
    print("Done")


if __name__ == "__main__":
    main()

Start processing YEAR = 2022
Profile input: /Users/yuanbao/Desktop/nogc_hcho/2022/all_profiles_2022.csv
TROPOMI dir: /Users/yuanbao/Desktop/TROPOMI/Data/hcho/2022
Saved: /Users/yuanbao/Desktop/nogc_hcho/2022/profile_data/daily_mean_profile.csv
Saved: /Users/yuanbao/Desktop/nogc_hcho/2022/profile_data/daily_column_from_profile.csv
Saved: /Users/yuanbao/Desktop/nogc_hcho/2022/profile_data/maxdoas_profile_for_smoothing.csv

Processing month 01
Skip month 01: MAXDOAS month folder missing: /Users/yuanbao/Desktop/nogc_hcho/2022/1

Processing month 02
Skip month 02: MAXDOAS month folder missing: /Users/yuanbao/Desktop/nogc_hcho/2022/2

Processing month 03
Skip month 03: MAXDOAS month folder missing: /Users/yuanbao/Desktop/nogc_hcho/2022/3

Processing month 04
Skip month 04: MAXDOAS month folder missing: /Users/yuanbao/Desktop/nogc_hcho/2022/4

Processing month 05
Skip month 05: MAXDOAS month folder missing: /Users/yuanbao/Desktop/nogc_hcho/2022/5

Processing month 06
Skip month 06: MAXDOAS mo